In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime

### Datos

In [21]:
sales = pd.read_csv('../data/sales.csv')
display(sales)

,id,doc_id,sale_date,total_cost,net_price,gross_price,profit,quantity,customer_id,product_id,product_class_id,route_id,warehouse_id
0,761095,gdl 455104,2025-04-30,1034.763158,2304.620410,2304.6204,1269.857252,10,14058,SAL2,nat,2.0,gdl
1,761096,gdl 455104,2025-04-30,3349.555935,5965.758504,5965.7585,2616.202569,5,14058,SAL30,nat,2.0,gdl
2,761097,gdl 455104,2025-04-30,3366.432542,5916.165713,5916.1657,2549.733171,6,14058,SBP18,nat,2.0,gdl
3,761098,gdl 455104,2025-04-30,1334.002702,0.000406,0.0004,-1334.002296,6,14058,SBP6,nat,2.0,gdl
4,761099,gdl 453004,2025-04-01,452.726995,799.324150,799.3242,346.597155,1,14060,8+18,nat,26.0,gdl
...,...,...,...,...,...,...,...,...,...,...,...,...,...
708012,761090,gdl 455104,2025-04-30,750.963953,1336.096598,1336.0966,585.132645,2,14058,PREM20,dmd,2.0,gdl
708013,761091,gdl 455104,2025-04-30,1263.054686,2228.772381,2228.7724,965.717695,2,14058,PREM40,dmd,2.0,gdl
708014,761092,gdl 455104,2025-04-30,394.354672,700.137959,700.1380,305.783287,3,14058,PREM6,dmd,2.0,gdl
708015,761093,gdl 455104,2025-04-30,4053.537940,7036.386122,7036.3861,2982.848182,6,14058,PUP40,dmd,2.0,gdl


In [22]:
routes = pd.read_csv('../data/routes.csv')
display(routes)

,route_number,route_name,commission_type,commission,notes,type_id,channel_id,warehouse_id
0,157,luis vallarta,f,10000.0,NaN,4,2,gdl
1,24,lorena colunga,v,0.0,NaN,4,2,snc
2,13,gerente puebla,v,0.0,ac,4,2,pue
3,3,vacante,v,0.0,NaN,4,2,snc
4,122,abraham,f,10000.0,ac,4,2,cdmx1
...,...,...,...,...,...,...,...,...
120,107,jorge,v,0.0,NaN,4,2,snc
121,110,perla,v,0.0,NaN,4,2,snc
122,100,gerente queretaro,v,0.0,NaN,4,2,snc
123,90,agente ecomerce,v,0.0,NaN,4,1,ecm


In [27]:
customers = pd.read_csv('../data/customers.csv')
display(customers)

,code,name,registration_date,credit_limit,credit_days,customer_type_id,opinion_leader
0,10420,leticia govea vazquez,2006-12-29,10000.0,15,8,False
1,10531,miguel angel dominguez galvez,2006-12-29,10000.0,15,5,False
2,10854,rodolfo gomez ortiz,2006-12-29,10000.0,30,5,False
3,10858,maria adela gutierrez salcido,2006-12-29,60000.0,30,8,False
4,10056,daniela alton zepeda,2006-12-29,30000.0,30,5,False
...,...,...,...,...,...,...,...
9118,93438,martha gabriela morales gomez,2026-03-18,10000.0,7,8,False
9119,93439,rosa isela sosa vazquez,2026-03-18,10000.0,7,8,False
9120,93440,erick rodriguez dominguez,2026-03-18,10000.0,7,5,False
9121,93464,victor manuel ramos gonzalez,2026-04-29,10000.0,7,5,False


### funciones auxiliares

In [33]:
def generate_moving_quarterly_report(sales_df: pd.DataFrame, 
                                     customers_df: pd.DataFrame, 
                                     target_year: int = None, 
                                     target_month: int = None) -> pd.DataFrame:
    sales = sales_df.copy()
    customers = customers_df.copy()
    
    sales['sale_date'] = pd.to_datetime(sales['sale_date'])
    customers['registration_date'] = pd.to_datetime(customers['registration_date'])
    
    sales['customer_id'] = sales['customer_id'].astype(str)
    customers['code'] = customers['code'].astype(str)
    
    if target_year is None:
        target_year = datetime.now().year
    if target_month is None:
        target_month = datetime.now().month
        
    target_start = pd.Timestamp(year=int(target_year), month=int(target_month), day=1)
    
    m1_start = target_start - pd.DateOffset(months=3)
    m2_start = target_start - pd.DateOffset(months=2)
    m3_start = target_start - pd.DateOffset(months=1)
    
    # Aquí se acota estrictamente el trimestre anterior
    window_sales = sales[(sales['sale_date'] >= m1_start) & (sales['sale_date'] < target_start)].copy()
    
    if window_sales.empty:
        return pd.DataFrame(columns=['year', 'month', 'route_id', 'customer_id', 'moving_quarterly_mean'])
        
    merged_data = window_sales.merge(
        customers[['code', 'registration_date']], 
        left_on='customer_id',
        right_on='code',
        how='left'
    )
    
    conditions = [
        merged_data['registration_date'] >= m3_start,
        (merged_data['registration_date'] >= m2_start) & (merged_data['registration_date'] < m3_start)
    ]
    choices = [1, 2]
    merged_data['denominator'] = np.select(conditions, choices, default=3)
    
    grouped = merged_data.groupby(['route_id', 'customer_id']).agg(
        total_net_price=('net_price', 'sum'),
        denominator=('denominator', 'first')
    ).reset_index()
    
    grouped['moving_quarterly_mean'] = grouped['total_net_price'] / grouped['denominator']
    grouped['year'] = target_start.year
    grouped['month'] = target_start.month
    
    columns_order = ['year', 'month', 'route_id', 'customer_id', 'moving_quarterly_mean']
    final_df = grouped[columns_order]
    final_df = final_df.sort_values(by=['year', 'month', 'route_id', 'customer_id']).reset_index(drop=True)
    
    return final_df

### Cruce de datos

In [34]:
sales_agg = generate_moving_quarterly_report(sales, customers)

display(sales_agg)

,year,month,route_id,customer_id,moving_quarterly_mean
0,2026,5,1.0,10451,812.927256
1,2026,5,1.0,10483,1189.241357
2,2026,5,1.0,10524,288.685350
3,2026,5,1.0,10828,103.237917
4,2026,5,1.0,11009,363.047596
...,...,...,...,...,...
3098,2026,5,157.0,70159,1176.206904
3099,2026,5,157.0,70160,2350.000000
3100,2026,5,157.0,70161,11116.551868
3101,2026,5,157.0,70163,2699.267480


In [36]:
sales_agg = sales_agg.merge(routes[['route_number','route_name', 'warehouse_id']], right_on='route_number', left_on='route_id', how='left')

sales_agg

,year,month,route_id,customer_id,moving_quarterly_mean,route_number,route_name,warehouse_id
0,2026,5,1.0,10451,812.927256,1,bodega guadalajara,gdl
1,2026,5,1.0,10483,1189.241357,1,bodega guadalajara,gdl
2,2026,5,1.0,10524,288.685350,1,bodega guadalajara,gdl
3,2026,5,1.0,10828,103.237917,1,bodega guadalajara,gdl
4,2026,5,1.0,11009,363.047596,1,bodega guadalajara,gdl
...,...,...,...,...,...,...,...,...
3098,2026,5,157.0,70159,1176.206904,157,luis vallarta,gdl
3099,2026,5,157.0,70160,2350.000000,157,luis vallarta,gdl
3100,2026,5,157.0,70161,11116.551868,157,luis vallarta,gdl
3101,2026,5,157.0,70163,2699.267480,157,luis vallarta,gdl


In [50]:
warehouses = routes['warehouse_id'].unique().tolist()
routes_dict = {}

for warehouse in warehouses:
    print(f'\n\nwarehouse {warehouse}\n')
    df = sales_agg[sales_agg['warehouse_id'] == warehouse]

    routes_dict[warehouse] = {}

    routes_list = df['route_id'].unique().tolist()

    print(f'route list -> {routes_list}\n')

    for route in routes_list:


        df_route = df[df['route_id'] == route]

        routes_dict[warehouse][route] = df_route

        print(f'route {route} dataframe successfuly created: {len(df_route)}')




warehouse gdl

route list -> [1.0, 2.0, 26.0, 41.0, 53.0, 55.0, 59.0, 64.0, 121.0, 145.0, 153.0, 157.0]

route 1.0 dataframe successfuly created: 98
route 2.0 dataframe successfuly created: 45
route 26.0 dataframe successfuly created: 61
route 41.0 dataframe successfuly created: 89
route 53.0 dataframe successfuly created: 41
route 55.0 dataframe successfuly created: 77
route 59.0 dataframe successfuly created: 88
route 64.0 dataframe successfuly created: 78
route 121.0 dataframe successfuly created: 86
route 145.0 dataframe successfuly created: 22
route 153.0 dataframe successfuly created: 74
route 157.0 dataframe successfuly created: 21


warehouse snc

route list -> [3.0, 24.0, 44.0, 50.0, 60.0, 74.0, 93.0, 115.0, 119.0, 154.0, 155.0]

route 3.0 dataframe successfuly created: 1
route 24.0 dataframe successfuly created: 2
route 44.0 dataframe successfuly created: 1
route 50.0 dataframe successfuly created: 1
route 60.0 dataframe successfuly created: 1
route 74.0 dataframe successfu

In [52]:
test = routes_dict['gdl'][53.0]
test

,year,month,route_id,customer_id,moving_quarterly_mean,route_number,route_name,warehouse_id
304,2026,5,53.0,11675,291.000000,53,alondra gonzalez,gdl
305,2026,5,53.0,11784,14717.458548,53,alondra gonzalez,gdl
306,2026,5,53.0,12819,2937.933556,53,alondra gonzalez,gdl
307,2026,5,53.0,12887,6491.996598,53,alondra gonzalez,gdl
308,2026,5,53.0,12930,5782.988398,53,alondra gonzalez,gdl
309,2026,5,53.0,13093,8756.466891,53,alondra gonzalez,gdl
310,2026,5,53.0,13212,2568.965493,53,alondra gonzalez,gdl
311,2026,5,53.0,14207,1158.894668,53,alondra gonzalez,gdl
312,2026,5,53.0,14344,2468.958852,53,alondra gonzalez,gdl
313,2026,5,53.0,14355,4233.731626,53,alondra gonzalez,gdl


In [48]:
routes_dict['gdl']

{157.0:       year  month  route_id customer_id  moving_quarterly_mean  route_number  \
 3082  2026      5     157.0       10928             456.154128           157   
 3083  2026      5     157.0       70033             133.340000           157   
 3084  2026      5     157.0       70091            1857.931128           157   
 3085  2026      5     157.0       70112            1663.562184           157   
 3086  2026      5     157.0       70147           18656.659454           157   
 3087  2026      5     157.0       70148           16139.010072           157   
 3088  2026      5     157.0       70149           26009.500020           157   
 3089  2026      5     157.0       70150             627.211352           157   
 3090  2026      5     157.0       70151            2801.914200           157   
 3091  2026      5     157.0       70152            7019.966572           157   
 3092  2026      5     157.0       70153           11592.613240           157   
 3093  2026      5   